### RNN (Recurrent Neural Network):

Recurrent Neural Network is a type of a neural network that have memory as a power to keep the context it learned from previous time-steps. We know that the hidden layer learns the patterns with the help of its neurons and activation function. So does the hidden state in the RNN but the difference is it uses the learnt stuff in next time step and so on.

In each time-step, it takes two input:

- the h(t-1) i.e. previous hidden state(context/memory/sentiment/meaning)it learned so far
- the current input x(t) in the respective time step

where the hidden state gives importance to each input, wh for previous hidden state and wi for the input in that time step.

It is always same for the whole dataset/batch, and after each backpropagation, it is updated.

At timestep t,
h(t) = (wh * h(t-1) + wi * x(t))

The main idea of the RNN is to consume the sequential data in such a way that it uses the context/meaning in the next timestep, keep summerizes the pattern from the start to the end to produce the required answer.

As the sequential data are the type of the data where the order matters, which means that the order in which the data exist gives meaning and will change/deflect entire meaning if order disrupts.

For example, for predicting the stock price in d5 (day5):
we need data in the order as d1 - d2 - d3 - d4 to summarize the pattern so that we can find the way the data is moving to make correct prediction.

In [268]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import re

In [269]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


Since the RNN doesnot understand the test/question and the asnwer as it is in words so we need to ocnvert the words into meaning to represent the word in number/ vector

In [270]:
# these are the labels of the data
df.iloc[:, 1].unique().tolist()

['Paris',
 'Berlin',
 'Harper-Lee',
 'Jupiter',
 '100',
 'Leonardo-da-Vinci',
 '8',
 'Au',
 '1945',
 'Nile',
 'Tokyo',
 'Albert-Einstein',
 '32',
 'Mars',
 'George-Orwell',
 'Pound',
 'Delhi',
 'Newton',
 '7',
 'CO2',
 '2',
 'Alexander-Graham-Bell',
 'Canberra',
 'Pacific-Ocean',
 '299,792,458m/s',
 'Portuguese',
 'Alexander-Fleming',
 'Ottawa',
 'Whale',
 'Hydrogen',
 'Everest',
 'NewYork',
 'vangogh',
 'H2O',
 'Rome',
 'Japan',
 'Armstrong',
 'Avocado',
 '6',
 'Yuan',
 'Jane-Austen',
 'Fe',
 'Diamond',
 'Asia',
 'George-Washington',
 'Parrot',
 'Simpsons',
 'VaticanCity',
 'Saturn',
 'Shakespeare',
 'Nitrogen',
 '206',
 'Mercury',
 'Moscow',
 'Benjamin-Franklin',
 'Canada',
 'Yellow',
 'February',
 'Biology',
 'China',
 'Nectar',
 'Night',
 'Seoul',
 'Edison',
 'Oxygen',
 '12',
 'Egypt',
 'Octopus',
 'Christmas',
 'Yen',
 'Basketball',
 'Australia',
 'MargaretThatcher',
 'Cheetah',
 'Madrid',
 'CharlesBabbage',
 'MexicoCity',
 'Piano',
 'ChristopherColumbus',
 'Pinocchio',
 'JamesCam

In [271]:
df.shape

(90, 2)

In [272]:
# tokenization of the text to tokens(smallest part of the text):
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', "", text)
    return text.split()


In [273]:
for i in df.iloc[:5, 0].to_numpy():
    print(i)
    print(tokenizer(i))
    print('=')

What is the capital of France?
['what', 'is', 'the', 'capital', 'of', 'france']
=
What is the capital of Germany?
['what', 'is', 'the', 'capital', 'of', 'germany']
=
Who wrote 'To Kill a Mockingbird'?
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']
=
What is the largest planet in our solar system?
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system']
=
What is the boiling point of water in Celsius?
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius']
=


In [274]:
# voccabulary
voccab = {
    '<UNK>' : 0
    }

In [281]:
# converts the words/tokens to voccabulary
def build_voccab(rows):
    for row in rows:
        tokenized_quetion = tokenizer(row[0])
        tokenized_answer = tokenizer(row[1])
        mereged_tokens = tokenized_quetion + tokenized_answer
        print(mereged_tokens)
        for token in mereged_tokens:
            if token not in voccab:
                voccab[token] = len(voccab)

In [283]:
build_voccab(df.iloc[:, ].to_numpy())

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harperlee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardodavinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'alberteinstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '1984',

In [284]:
voccab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harperlee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardodavinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'alberteinstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'georgeorwell': 68,
 'currency': 69,
 'united': 

In [278]:
# embedding the tokens to fixed size of vecotrs of numbers

In [279]:
# train the rnn

In [280]:
# eval the rnn